Project.4. Statistical Analysis: A/B Testing 

Phase 1: Data Cleaning

In [1]:
pip install scipy

   ---------------------------------------- 0.0/37.3 MB ? eta -:--:--
   - -------------------------------------- 1.3/37.3 MB 8.8 MB/s eta 0:00:05
   -- ------------------------------------- 2.4/37.3 MB 6.2 MB/s eta 0:00:06
   --- ------------------------------------ 3.7/37.3 MB 6.3 MB/s eta 0:00:06
   ----- ---------------------------------- 4.7/37.3 MB 5.9 MB/s eta 0:00:06
   ------ --------------------------------- 6.0/37.3 MB 6.1 MB/s eta 0:00:06
   ------- -------------------------------- 7.1/37.3 MB 5.8 MB/s eta 0:00:06
   --------- ------------------------------ 8.9/37.3 MB 6.1 MB/s eta 0:00:05
   ---------- ----------------------------- 10.2/37.3 MB 6.2 MB/s eta 0:00:05
   ------------ --------------------------- 11.5/37.3 MB 6.2 MB/s eta 0:00:05
   ------------- -------------------------- 12.8/37.3 MB 6.2 MB/s eta 0:00:04
   --------------- ------------------------ 14.2/37.3 MB 6.2 MB/s eta 0:00:04
   ---------------- ----------------------- 15.5/37.3 MB 6.2 MB/s eta 0:00:04
 


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: C:\Users\Aishwarya Sonawane\AppData\Local\Programs\Python\Python314\python.exe -m pip install --upgrade pip


In [8]:
pip install statsmodels

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: C:\Users\Aishwarya Sonawane\AppData\Local\Programs\Python\Python314\python.exe -m pip install --upgrade pip


In [17]:
import pandas as pd
import numpy as np

df = pd.read_csv('ab_data.csv')

# Problem 1: Mismatched rows
# Some control users saw new_page and vice versa — these are errors, remove them
mismatch = df[
    ((df['group']=='control')   & (df['landing_page']=='new_page')) |
    ((df['group']=='treatment') & (df['landing_page']=='old_page'))
]
df_clean = df[df.index.isin(mismatch.index)]
# Removed 3,893 mismatched rows

# Problem 2: Duplicate user_ids (3,894 duplicates)
df_clean = df_clean.drop_duplicates(subset='user_id', keep='first')
# Final clean dataset: 290,584 rows
print(df)

        user_id                   timestamp      group landing_page  converted
0        851104  2017-01-21 22:11:48.556739    control     old_page          0
1        804228  2017-01-12 08:01:45.159739    control     old_page          0
2        661590  2017-01-11 16:55:06.154213  treatment     new_page          0
3        853541  2017-01-08 18:28:03.143765  treatment     new_page          0
4        864975  2017-01-21 01:52:26.210827    control     old_page          1
...         ...                         ...        ...          ...        ...
294473   751197  2017-01-03 22:28:38.630509    control     old_page          0
294474   945152  2017-01-12 00:51:57.078372    control     old_page          0
294475   734608  2017-01-22 11:45:03.439544    control     old_page          0
294476   697314  2017-01-15 01:20:28.957438    control     old_page          0
294477   715931  2017-01-16 12:40:24.467417  treatment     new_page          0

[294478 rows x 5 columns]


Phase 2: Group Statistics

In [18]:
control   = df_clean[df_clean['group'] == 'control']
treatment = df_clean[df_clean['group'] == 'treatment']

control_cr   = control['converted'].mean()    # → 12.04%
treatment_cr = treatment['converted'].mean()  # → 11.88%
uplift = (treatment_cr - control_cr) / control_cr * 100  # → -1.31%

Phase 3: Hypothesis Testing
Test 1 — Chi-Square Test (for conversion rate)

In [19]:
from scipy import stats

contingency = pd.crosstab(df_clean['group'], df_clean['converted'])
chi2, p_value, dof, expected = stats.chi2_contingency(contingency)
# chi2 = 1.7036, p-value = 0.1918

Test 2 — Z-Test for Proportions

In [20]:
from statsmodels.stats.proportion import proportions_ztest

count = np.array([control['converted'].sum(), treatment['converted'].sum()])
nobs  = np.array([len(control), len(treatment)])
z_stat, p_z = proportions_ztest(count, nobs)
# z = 1.3109, p-value = 0.1899

Test 3 — 95% Confidence Intervals

In [24]:
from scipy.stats import t as t_dist
n_c = len(control)
n_t = len(treatment)

m_c = control['converted'].mean()
m_t = treatment['converted'].mean()

s_c = control['converted'].std()
s_t = treatment['converted'].std()
ci_c = t_dist.interval(0.95, n_c-1, loc=m_c, scale=s_c/np.sqrt(n_c))
ci_t = t_dist.interval(0.95, n_t-1, loc=m_t, scale=s_t/np.sqrt(n_t))


Test 4 — Cohen's d (Effect Size)


In [22]:
pooled_std = np.sqrt((s_c**2 + s_t**2) / 2)
cohens_d   = (m_t - m_c) / pooled_std
# Cohen's d = -0.004864 → NEGLIGIBLE effect

Phase 4: Daily Trend Analysis

In [23]:
df_clean['timestamp'] = pd.to_datetime(df_clean['timestamp'])
df_clean['date'] = df_clean['timestamp'].dt.date

daily = df_clean.groupby(['date','group'])['converted'].mean() * 100

Statistical Summary :
Metric                      Value
Control Conversion Rate  	12.04%
Treatment Conversion Rate	11.88%
Uplift	                    -1.31%
Chi-Square statistic       	1.7036
p-value                 	0.1918
Z-statistic             	1.3109
95% CI Control	            11.87%, 12.21%
95% CI Treatment	        11.71%, 12.05%
Decision                    Fail to reject H₀ — Not Significant

Business Recommendations
Based on the statistical results:

Do NOT launch the new page — it performs slightly worse (-1.31%) with no statistical evidence the difference is real

Cost saved — with 145k users tested and no gain, deploying the new page would risk losing 1.31% of conversions with zero upside

Redesign required — the new page needs fundamental improvements, not cosmetic ones

Run longer test —  Cohen's d = 0.005 means even with infinite data, the effect is practically meaningless

Segment the test — check if the new page works better for specific user groups (device type, geography, new vs returning users) before fully discarding it